# Forecast of the harmonic residual current – network model of a busbar

An **optional part** of groundfault: it builds on the **result files** of the single evaluations
(`evaluate_measurement.ipynb`, section 7) and needs nothing else of the evaluation. The **network model** of the
earth-fault loop at a busbar per harmonic order is calibrated with the staged fault tests, updated with every earth
fault recorded at the busbar and checked; from harmonic voltages – e.g. the 95 % quantiles of a power-quality
measurement – it forecasts the **harmonic residual current** I_H and the residual current I_RES of an earth fault at
the busbar or behind a line. Reference copy: `templates/network_model_forecast_DEFAULT.ipynb`. Method and
equivalent circuits: online documentation, *Forecast*.

1. **Calibration** with the staged fault tests of the campaigns (fault current measured): sequence impedances
   Z_0, Z_1, Z_2 per order from the changes of the busbar voltages, Z_q = (Z_0 + Z_1 + Z_2)/3; at the orders of the
   zero sequence also the source that only exists during the fault (saturating coils and transformers).
2. **Checks between the tests** – the deviations show the margin a forecast needs.
3. **Recorded earth faults** (busbar voltages, e.g. of a disturbance recorder): Z_1, Z_2, Z_q updated, the fault
   current estimated (with coil currents by the node rule at every order) and the forecast checked at each of them.
4. **Checks with other measurements**, e.g. a staged fault test at a location in the network.
5. **Forecast** of I_H and I_RES (one value per order or a time series; at the busbar or behind `LINE`).
6. **Save** the model file and the tables in the forecast folder of the work folder.

In [ ]:
%matplotlib inline
import datetime
import json
import sys
import warnings
from pathlib import Path

LANGUAGE = "en"                # output language of tables, hints and figures: "en" | "de"
try:
    import groundfault as gf
except ImportError:            # not installed: use the source folder of a clone (…/groundfault/src)
    for p in [Path.cwd(), *Path.cwd().parents]:    # ./src, ./groundfault/src, ./*/groundfault/src
        try:
            candidates = [p / "src", p / "groundfault" / "src", *sorted(p.glob("*/groundfault/src"))]
        except OSError:
            candidates = [p / "src", p / "groundfault" / "src"]
        hits = [c for c in candidates if (c / "groundfault" / "__init__.py").is_file()]
        if hits:
            sys.path.insert(0, str(hits[0]))
            break
    import groundfault as gf
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from groundfault import forecast as fc
from groundfault import (campaign, corrections, demo, diagnostics, disturbances, earthing, equivalent_source,
                         explorer, i18n, interactive, layout, legacy, register, report, rms, sites, transformer)

i18n.set_language(LANGUAGE)
# work folder = this folder or the first folder above it with a campaigns folder ("campaigns" or
# "01_Messkampagnen"); its name gives the folder layout, without one the layout of LANGUAGE is used
WORK_FOLDER, LAYOUT = layout.find_work_folder(Path.cwd(), default=LANGUAGE)
WORK_FOLDER = Path(WORK_FOLDER)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: "%.4g" % x)
report.activate_style()        # notebook preview = typeface of the exported figures


def show(df):
    """Table with column names and codes in the output language."""
    display(i18n.localize(df))


print("groundfault %s · work folder: %s · layout: %s · font of the figures: %s"
      % (gf.__version__, WORK_FOLDER, LAYOUT.language, report.used_font()))

## 1 · Configuration

In [ ]:
NETWORK   = "Demo substation"            # name of the network model (substation, busbar)
DEMO      = True                         # create the demo campaign, a recorded earth fault and their result files if missing
CAMPAIGNS = ["2026-01-15_Demo"]          # campaigns whose staged fault tests calibrate the model (from the register)
TESTS     = None                         # None = the tests of CAMPAIGNS from the register; or a list of dict(campaign=…, measurement=…, Z=…)
# Recorded earth faults at the busbar (update): result files of evaluate_measurement.ipynb with the busbar voltages
# before and during the fault – no fault current needed; with coil currents also the fault current per order
FAULTS = [
    dict(campaign="2026-01-15_Demo", measurement="SUB_fault_10_000kHz"),
]
# Measurements to check the model with, e.g. a staged fault test at a location in the network:
# dict(campaign=…, measurement=<busbar recording>, Z=…, current=<recording with the fault current>, line=LINE)
CHECKS = []
# optional per entry: name="…" (default: the measurement), meta=dict(…) (e.g. load, PV infeed, switching state),
# transformer="…" (transformer recording of the same earth fault by a second recorder: its upstream impedance Z_up),
# line=dict(…) (a test in the network: the line to its fault location)

# ---------------------------------------------------------------- model
ORDERS        = "auto"            # "auto" = odd orders ≥ 3 with at least MIN_CURRENT_A in the test, otherwise e.g. [3, 5, 7, 11, 13]
MIN_CURRENT_A = 0.3               # smallest current of an order for "auto" in A
HOW           = "median"          # Z_q over the earth faults: "median" | "mean" | "latest" (the last one added)
TWO_SOURCE    = "zero_sequence"   # orders with a source that arises in the fault: "zero_sequence" (3, 9, 15 …) | [orders] | None

# ---------------------------------------------------------------- forecast
VOLTAGES          = {3: 30.0, 5: 250.0, 7: 130.0, 11: 40.0, 13: 30.0}   # demo values: 95 % quantiles in V (line-to-earth)
VOLTAGE_REFERENCE = "LE"          # "LE" = line-to-earth | "LL" = line-to-line (divided by √3; no zero sequence)
CSV_OPTIONS       = dict(sep=";", decimal=",", index_col=0)   # reading a CSV file (pandas.read_csv): one column per order
QUANTILE          = 0.95          # quantile of a time series
Z_F               = 0.0           # fault impedance in Ω (0 = fault without resistance)
LINE              = dict(name="Demo feeder, station 4", segments=[         # demo: 4 km cable, 1.5 km overhead line
    dict(length_km=4.0, R1=0.125, X1=0.11, R0=0.50, X0=0.40, C1_uF=0.30),
    dict(length_km=1.5, R1=0.30,  X1=0.35, R0=0.45, X0=1.20, C1_uF=0.01)])
# per-length values at 50 Hz (Ω/km, µF/km) give R constant, X ∝ m – not on the safe side; at the orders of interest give
# values of their own: per segment orders={5: dict(R1=…, X1=…, R0=…, X0=…)} (Ω/km at 250 Hz, X not divided by 5) or for
# the whole line loop={5: Z_qL} (Ω, e.g. measured with MODEL.line_from_fault(<recorded earth fault behind the line>))

# ---------------------------------------------------------------- margins and limits – defined by you
# groundfault contains no margins or limits. Not set (the values below), the forecast is shown without them.
MARGIN_A      = 0.0     # margin per order in A: a number or {order: A} (0.0 = none), e.g. from the checks of section 3
FUNDAMENTAL_A = None    # fundamental residual current in A (None = median of √(I_w² + I_v²) of the tests)
LIMITS        = ()      # limits of the residual current in A, e.g. (I_1, I_2)

## 2 · Calibration with the staged fault tests

Per test and order the **B method**: Z_s = 3·p_s·ΔU_s/I (s = zero, positive, negative sequence; ΔU = change of the
busbar voltage during the fault, I = fault current, p_s = phase factor of the faulted phase). The source is the
pre-fault voltage of the faulted phase, U_q = −Σ p_s·U_pre,s. At the orders of `TWO_SOURCE` the tests with different
additional impedances also separate the source U_s that arises in the fault (`two_source`); with only one additional
impedance the measured current is used for every fault impedance (`constant`). With coil currents in the tests the
admittance of the network without the coils, Y_0,net = (I − I_coil)/(3·ΔU_0), is kept for the node rule (section 4).

In [ ]:
for i, e in enumerate(FAULTS + CHECKS):     # known keys only (a misspelt key would be ignored)
    legacy.check_keys("FAULTS" if i < len(FAULTS) else "CHECKS", e, fc.ENTRY_KEYS + ("current",))

if DEMO:                         # demo: campaign, a recording of the busbar voltages and coil currents, result files
    for c in CAMPAIGNS:
        if not (WORK_FOLDER / LAYOUT.campaigns / c).exists():
            demo.make_campaign(str(WORK_FOLDER), c, layout=LAYOUT.language)
    for e in [*FAULTS, *[dict(campaign=c, measurement=t["measurement"], Z=t["Z"]) for c in CAMPAIGNS
                         for t in fc.campaign_tests(WORK_FOLDER, c)]]:
        folder = WORK_FOLDER / LAYOUT.campaigns / e["campaign"]
        cfg = folder / LAYOUT.raw / ("SUB" if LAYOUT.language == "en" else "UW") / (e["measurement"] + ".cfg")
        if e in FAULTS and not cfg.exists():            # recorded earth fault: busbar voltages and coil currents
            demo.make_substation_recording(str(cfg), 0.0, drift=demo.polar(1.15, 20), fault_current=False, seed=7,
                                           start=datetime.datetime(2026, 3, 2, 14, 7))
        if not Path(fc.result_path(WORK_FOLDER, e["campaign"], e["measurement"])).exists():
            with open(str(cfg).replace(".cfg", "_truth.json"), encoding="utf-8") as f:
                b = json.load(f)["intervals"]
            r_ = gf.evaluate(gf.open_recording(str(cfg)),
                             gf.Intervals(*(tuple(b[k]) for k in ("steady_state", "pre_fault", "post_fault"))),
                             z_fault_ohm=e.get("Z"))
            r_.save_json(LAYOUT.result_path(folder, e["measurement"]))
if TESTS is None:
    TESTS = [t for c in CAMPAIGNS for t in fc.campaign_tests(WORK_FOLDER, c)]
display(Markdown("**Staged fault tests** (`TESTS`)"))
show(pd.DataFrame(TESTS))
MODEL = fc.NetworkModel(NETWORK)
for msg in fc.add_entries(MODEL, WORK_FOLDER, TESTS, "test", orders=ORDERS, min_current_A=MIN_CURRENT_A):
    print("ℹ", msg)
if MODEL.tests:
    show(MODEL.table())
    display(Markdown("**Model per order**"))
    show(MODEL.source_table(how=HOW, two_source=TWO_SOURCE))
    for T in MODEL.source_terms(how=HOW, two_source=TWO_SOURCE).values():
        for h in T.hints:
            print("ℹ", h)
else:
    print("No calibration – the model needs at least one staged fault test with the fault current (TESTS).")

## 3 · Checks between the tests

Every test is forecast from its own pre-fault voltage and additional impedance with the model of other tests:
`loo` = model of all other tests, `pairwise` = model of one other test (every pair), `chain` = Z_0 of one test, Z_q
from the busbar voltages of a second one (as a recorded earth fault), forecast for a third. `deviation_A` = forecast
− measured: **negative values are underestimates**. The largest underestimate per order of `loo` – a model of the
other tests, like the forecast with all of them – is a basis for your margin `MARGIN_A`; `pairwise` and `chain` show
how far a single test or an update with a recorded earth fault carries (with one test the orders of the zero sequence
keep the current of that test). The row `I_H` is the whole harmonic content (orders not modelled included).

In [ ]:
CROSS_CHECKS = pd.DataFrame()
if len(MODEL.tests) >= 2:
    modes = ["loo", "pairwise"] + (["chain"] if len(MODEL.tests) >= 3 else [])
    CROSS_CHECKS = pd.concat([MODEL.cross_check(0.0, mode, how=HOW, two_source=TWO_SOURCE) for mode in modes],
                             ignore_index=True)
    summary = (CROSS_CHECKS.assign(order=CROSS_CHECKS["order"].astype(str))
               .groupby(["order", "mode"], sort=False)
               .agg(n=("deviation_A", "size"), deviation_min_A=("deviation_A", "min"),
                    deviation_max_A=("deviation_A", "max"), ratio_min=("ratio", "min"), ratio_max=("ratio", "max"))
               .reset_index())
    show(summary)
    under = CROSS_CHECKS[CROSS_CHECKS["order"] != "I_H"].groupby(["mode", "order"])["deviation_A"].min()
    for mode in [m for m in modes if m in under.index.get_level_values(0)]:   # per check, without margin
        print(i18n.format_text("Largest underestimate per order without margin ({}):",
                               i18n.value_label("mode", mode)),
              ", ".join(i18n.format_text("{}: {:.2f} A", m, min(v, 0.0)) for m, v in under[mode].items()))
    display(Markdown("**All other tests (loo)**"))
    show(CROSS_CHECKS[CROSS_CHECKS["mode"] == "loo"])
else:
    print("The checks need at least two staged fault tests.")

## 4 · Recorded earth faults

An earth fault of the network recorded at the busbar – line-to-earth voltages before and during the fault, evaluated
with `evaluate_measurement.ipynb` (pre-fault and steady-state interval; a fault current is not needed). With Z_0 of
the tests: Z_s = Z_0·p_s·ΔU_s/ΔU_0. The **fault current** of the earth fault (`I_estimated_A`): with the coil currents
in the recording and in the tests by the **node rule** of the zero sequence, I_F = 3·Y_0,net·ΔU_0 + I_coil – at every
order, also where a source arises in the fault –, otherwise 3·|ΔU_0|/|Z_0|. The orders of `TWO_SOURCE` are not
updated. A recording with the line currents of the transformer – or `transformer=` – also gives the upstream impedance
Z_up and with it the MV network Z_down. Each earth fault then **checks the forecast**: the model of all other earth
faults forecasts it from its own pre-fault voltage with the fault impedance 0 (the fault impedance of a natural earth
fault is not known – its current can only be smaller).

In [ ]:
FAULT_CHECKS = pd.DataFrame()
if FAULTS and not MODEL.tests:
    print("⚠ a recorded earth fault needs Z_0 of a calibration (section 2)")
elif FAULTS:
    for msg in fc.add_entries(MODEL, WORK_FOLDER, FAULTS, "fault", two_source=TWO_SOURCE):
        print("ℹ", msg)
if MODEL.faults:
    tab = MODEL.table()
    show(tab[tab["kind"] == "fault"])
    display(Markdown("**|Z_q| of the earth faults ÷ median of the tests**"))
    ref = tab[tab["kind"] == "test"].groupby("order")["Zq_Ohm"].median()
    show(tab[tab["kind"] == "fault"].pivot(index="order", columns="name", values="Zq_Ohm").div(ref, axis=0))
    if any(c.currents for c in MODEL.faults):
        FAULT_CHECKS = MODEL.fault_check(0.0, how=HOW, two_source=TWO_SOURCE)
        display(Markdown("**Forecast (fault impedance 0, without margin) ÷ estimated fault current**"))
        show(FAULT_CHECKS)
else:
    print("No recorded earth fault in the model (FAULTS).")

## 5 · Checks with other measurements

A measurement that is not part of the model – e.g. a staged fault test at a location in the network: the busbar
voltages of the substation recording (`measurement`), the fault current of the recording at the fault location
(`current`, magnitudes per order) and the line to the fault location (`line`). The model forecasts the fault current
from the pre-fault voltage of the busbar, the fault impedance `Z` and the line, Z_q + (2·Z_1L + Z_0L)/3 + Z_F. For an
entry with a line the busbar voltages of the same fault also measure the line per order (`MODEL.line_from_fault`):
Z_qL,m = −U_ph,bus,m/I_F,m − Z_F with the fault current from the busbar (node rule, otherwise 3·ΔU_0/Z_0 at the orders
without a source in the fault). `LINE_MEASURED` holds the values – as `loop=` of a line they give the forecast its own
values per order.

In [ ]:
VERIFY = pd.DataFrame()
LINE_MEASURED, LINE_TABLE = {}, pd.DataFrame()
if CHECKS and MODEL.tests:
    parts, rows = [], []
    for e in CHECKS:
        try:
            fd = fc.check_data(WORK_FOLDER, e)
            parts.append(MODEL.verify(fd, 0.0, how=HOW, two_source=TWO_SOURCE))
            if fd.line is not None:          # the line per order from the busbar voltages of the same fault
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    z = LINE_MEASURED[fd.name] = MODEL.line_from_fault(fd, fd.Z_F or 0.0, two_source=TWO_SOURCE)
                for w in caught:
                    print("ℹ", w.message)
                for m, v in sorted(z.items()):
                    g = fd.line.Zq(m)
                    rows.append(dict(target=fd.name, order=m, line_values=fd.line.source(m), Z_line_Ohm=abs(g),
                                     R_measured_Ohm=v.real, X_measured_Ohm=v.imag, Z_measured_Ohm=abs(v),
                                     ratio=abs(v) / abs(g) if abs(g) else float("nan")))
        except (OSError, ValueError) as ex:
            print("⚠", ex)
    if parts:
        VERIFY = pd.concat(parts, ignore_index=True)
        show(VERIFY)
    if rows:
        LINE_TABLE = pd.DataFrame(rows)
        display(Markdown("**Line per order: given (`line`) and measured from the busbar voltages of the fault (`LINE_MEASURED`)**"))
        show(LINE_TABLE)
else:
    print("ℹ No further measurements (CHECKS).")

## 6 · Forecast of the harmonic residual current

Harmonic voltages of the busbar per order (line-to-earth, the largest of the three phases): one value per order –
e.g. the 95 % quantile of a power-quality measurement – or a time series (CSV file, one column per order such as
`U_H5`; the last number of a column name is the order). Per order |I_F,m| = (|U_m| + |U_s,m|)/|Z_q,m + Z_qL,m + Z_F|
+ margin (Z_qL = 0 at the busbar, (2·Z_1L + Z_0L)/3 behind `LINE`), then I_H = √(Σ I_F,m² + I_rest²) with I_rest =
harmonic content of the tests outside the forecast orders and I_RES = √(I_1² + I_H²). A time series gives the
quantile `QUANTILE` of the time series of I_H and I_RES.

In [ ]:
FORECAST = FORECAST_TABLE = None
if VOLTAGES is None:
    print("ℹ Forecast skipped – VOLTAGES is not set (section 1).")
elif not MODEL.calibrations:
    print("ℹ Forecast skipped – the model contains no calibration (section 2).")
else:
    U = pd.read_csv(VOLTAGES, **CSV_OPTIONS) if isinstance(VOLTAGES, (str, Path)) else VOLTAGES
    places = [(None, "busbar")] + ([(LINE, LINE.get("name") or "LINE")] if LINE else [])
    FORECASTS, tables = {}, []
    for line, place in places:
        f = FORECASTS[place] = MODEL.forecast(U, Z_F=Z_F, margin_A=MARGIN_A, how=HOW, two_source=TWO_SOURCE,
                                              reference=VOLTAGE_REFERENCE, fundamental_A=FUNDAMENTAL_A, line=line)
        series = len(f.currents) > 1
        tab = f.table(QUANTILE if series else None)
        tab.insert(0, "location", place)
        tables.append(tab)
        display(Markdown("**%s**" % place))
        show(tab)
        for h in f.hints:
            print("ℹ", h)
        value = tab.set_index("order")["I_forecast_A"]
        print(i18n.format_text("Forecast{}: I_H = {:.2f} A, I_RES = {:.2f} A (fault impedance {:g} Ω)",
                               i18n.format_text(" (quantile {:g} of {} time steps)",
                                                QUANTILE, len(f.currents)) if series else "",
                               value["I_H"], value["I_RES"], abs(complex(Z_F))))
        if LIMITS:
            for limit in LIMITS:
                print(i18n.format_text("I_RES = {:.2f} A {} {:g} A", value["I_RES"],
                                       "≤" if value["I_RES"] <= limit else ">", limit))
    FORECAST = FORECASTS[places[0][1]]                   # at the busbar
    FORECAST_TABLE = pd.concat(tables, ignore_index=True)
    if not MARGIN_A:
        print("ℹ Without a margin – MARGIN_A is not set (section 1); section 3 shows the deviations of the model.")
    if not LIMITS:
        print("ℹ No comparison with limits – LIMITS is not set (section 1).")

## 7 · Save the model and export the tables

In [ ]:
OUT = Path(fc.forecast_folder(WORK_FOLDER, NETWORK))
if MODEL.calibrations:
    path = OUT / (report.slug(NETWORK) + "_network_model.json")
    MODEL.save(str(path))
    TABLES = {"forecast_calibrations": MODEL.table(),
              "forecast_model": MODEL.source_table(how=HOW, two_source=TWO_SOURCE),
              "forecast_checks": pd.concat([CROSS_CHECKS, VERIFY], ignore_index=True),
              "forecast_faults": FAULT_CHECKS, "forecast": FORECAST_TABLE}
    paths = report.export_tables({k: v for k, v in TABLES.items() if v is not None and len(v)}, str(OUT),
                                 prefix=report.slug(NETWORK) + "_", formats=("csv", "md"))
    print("model file:", path)
    print("%d table files in %s" % (len(paths), OUT))
    # later: fc.NetworkModel.load(path) – add earth faults with add_fault, forecast with forecast